In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import zipfile
import io
zip_file_name = 'Dataset-Mini.zip'
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall()


In [ ]:
train_path = "Dataset-Mini/Train"
val_path   = "Dataset-Mini/Validation"
test_path  = "Dataset-Mini/Test"

def load_data(path):
    X, y = [], []

    for label, cls in enumerate(["Real", "Fake"]):
        cls_path = os.path.join(path, cls)
        for img_name in os.listdir(cls_path):
            img = cv2.imread(os.path.join(cls_path, img_name))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img = cv2.resize(img, (224, 224))
            X.append(img)
            y.append(label)

    X = np.array(X) / 255.0
    X = X.reshape(-1, 224, 224, 1)
    y = np.array(y)

    return X, y


In [ ]:
X_train, y_train = load_data(train_path)
X_val, y_val     = load_data(val_path)
X_test, y_test   = load_data(test_path)


In [ ]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,1)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val)
)


In [ ]:
train_acc = model.evaluate(X_train, y_train)[1] * 100
val_acc   = model.evaluate(X_val, y_val)[1] * 100
test_acc  = model.evaluate(X_test, y_test)[1] * 100

print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)


In [ ]:
import matplotlib.pyplot as plt

def visualize_predictions(X, y, model, n=5):
    preds = model.predict(X)
    for i in range(n):
        plt.imshow(X[i].reshape(224,224), cmap='gray')
        plt.title(f"True: {'Real' if y[i]==0 else 'Fake'}, Pred: {'Real' if preds[i]<0.5 else 'Fake'}")
        plt.show()

visualize_predictions(X_test, y_test, model)
# Commented out broken function call to prevent NameError:
# show_predictions(test_path, model, num_images=20)
